# YOLOv8 + CBAM + DANN — Domain Adaptive Object Detection

**Architecture:**
- **YOLOv8n** — base detector
- **CBAM** — Channel & Spatial Attention on all C2f blocks
- **DANN** — Gradient Reversal Layer + Domain Classifier for domain adaptation

**Dataset:** NEU-GC10 steel surface defects (source: original, target: augmented to simulate different camera/lighting)

**Make sure to set Runtime → Change runtime type → GPU (T4)**

## 1. Install Dependencies

In [ ]:
!pip install -q ultralytics torch torchvision tqdm pyyaml opencv-python-headless matplotlib roboflow

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Write Model Code

In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from ultralytics.nn.modules import Conv, C2f
from ultralytics.nn.tasks import DetectionModel, BaseModel
from ultralytics.utils.torch_utils import initialize_weights
from ultralytics.cfg import get_cfg
import yaml

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction_ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction_ratio, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction_ratio, channels, bias=False)
        )

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x).view(x.size(0), -1))
        max_out = self.fc(self.max_pool(x).view(x.size(0), -1))
        return torch.sigmoid(avg_out + max_out).view(x.size(0), -1, 1, 1)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size//2)

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return torch.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))

class CBAM(nn.Module):
    def __init__(self, channels, reduction_ratio=16, kernel_size=7):
        super().__init__()
        self.channel_attention = ChannelAttention(channels, reduction_ratio)
        self.spatial_attention = SpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.channel_attention(x)
        x = x * self.spatial_attention(x)
        return x

class GradientReversalLayer(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.clone()

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None

class DomainClassifier(nn.Module):
    def __init__(self, input_channels, hidden_size=1024):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(input_channels, hidden_size),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(hidden_size // 2, 1)
        )

    def forward(self, x, alpha=1.0):
        x = GradientReversalLayer.apply(x, alpha)
        x = self.pool(x).view(x.size(0), -1)
        return self.classifier(x)

class CBAMC2f(C2f):
    def __init__(self, c1, c2, *args, **kwargs):
        super().__init__(c1, c2, *args, **kwargs)
        self.cbam = CBAM(c2)

    def forward(self, x):
        x = super().forward(x)
        return self.cbam(x)

class DomainAdaptiveYOLOv8(DetectionModel):
    def __init__(self, cfg='yolov8n.yaml', ch=3, nc=None, task='detect'):
        super().__init__(cfg, ch, nc)

        if not hasattr(self, 'args'):
            self.args = get_cfg()

        self._replace_c2f_with_cbam()

        self.domain_classifier = None
        self._initialize_domain_classifier()

        self._cached_features = None
        self._feature_hook = self.model[-2].register_forward_hook(self._hook_features)

        initialize_weights(self)

    def _hook_features(self, module, input, output):
        if isinstance(output, (list, tuple)):
            self._cached_features = output[-1]
        else:
            self._cached_features = output

    def _replace_c2f_with_cbam(self):
        replaced_count = 0
        for i, m in enumerate(self.model):
            if isinstance(m, C2f) and not isinstance(m, CBAMC2f):
                c1 = m.cv1.conv.in_channels
                c2 = m.cv2.conv.out_channels
                n = len(m.m)
                shortcut = m.m[0].add if n > 0 else False
                g = m.m[0].cv2.conv.groups if n > 0 else 1
                e = m.c / c2 if c2 > 0 else 0.5
                new_module = CBAMC2f(c1, c2, n, shortcut, g, e)
                new_module.load_state_dict(m.state_dict(), strict=False)
                for attr in ('i', 'f', 'type', 'np'):
                    if hasattr(m, attr):
                        setattr(new_module, attr, getattr(m, attr))
                self.model[i] = new_module
                replaced_count += 1
        print(f'Replaced {replaced_count} C2f modules with CBAMC2f (with CBAM attention)')

    def _initialize_domain_classifier(self):
        device = next(self.parameters()).device
        dummy_input = torch.randn(1, 3, 640, 640, device=device)
        with torch.no_grad():
            features = self._extract_features(dummy_input)
            if features is not None:
                self.domain_classifier = DomainClassifier(features.shape[1])
            else:
                self.domain_classifier = DomainClassifier(256)

    def _extract_features(self, x):
        y = []
        for i, m in enumerate(self.model):
            if m.f != -1:
                x = y[m.f] if isinstance(m.f, int) else [x if j == -1 else y[j] for j in m.f]
            x = m(x)
            y.append(x if m.i in self.save else None)
            if i == len(self.model) - 2:
                if isinstance(x, (list, tuple)):
                    return x[-1]
                return x
        return None

    def forward(self, x, alpha=1.0, return_domain=False):
        detections = super().forward(x)
        if return_domain and self.domain_classifier is not None:
            features = self._cached_features
            if features is not None:
                domain_pred = self.domain_classifier(features, alpha)
                return detections, domain_pred
            else:
                batch_size = x.shape[0]
                domain_pred = torch.zeros(batch_size, 1, device=x.device)
                return detections, domain_pred
        return detections

    def export(self, **kwargs):
        """Export model to ONNX format"""
        domain_classifier = self.domain_classifier
        self.domain_classifier = None
        self._feature_hook.remove()

        exported_model = super().export(**kwargs)

        self.domain_classifier = domain_classifier
        self._feature_hook = self.model[-2].register_forward_hook(self._hook_features)

        return exported_model

## 3. Write Training Script

In [ ]:
%%writefile train.py
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
import yaml
import argparse
from tqdm import tqdm

from model import DomainAdaptiveYOLOv8
from ultralytics import YOLO
from ultralytics.data import build_dataloader, build_yolo_dataset
from ultralytics.utils import LOGGER
from ultralytics.utils.torch_utils import select_device
from torch.nn.parallel import DataParallel, DistributedDataParallel

def de_parallel(model):
    return model.module if isinstance(model, (DataParallel, DistributedDataParallel)) else model

try:
    from ultralytics.utils.loss import v8DetectionLoss
    HAS_DETECTION_LOSS = True
except ImportError:
    HAS_DETECTION_LOSS = False
    LOGGER.warning('Could not import v8DetectionLoss')


class DomainAdaptiveTrainer:
    def __init__(self, source_data, target_data, epochs=50, batch_size=16,
                 img_size=640, device='', workers=4, save_dir='runs/domain_adaptive',
                 pretrained='yolov8n.pt', lr=0.001, val_interval=5):
        self.epochs = epochs
        self.batch_size = batch_size
        self.img_size = img_size
        self.lr = lr
        self.val_interval = val_interval
        self.pretrained = pretrained
        self.device = select_device(device)
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.workers = workers

        with open(source_data) as f:
            self.source_cfg = yaml.safe_load(f)
        with open(target_data) as f:
            self.target_cfg = yaml.safe_load(f)

        self.setup_model()
        self.setup_dataloaders()
        self.setup_training()

    def setup_model(self):
        nc = len(self.source_cfg['names'])
        self.model = DomainAdaptiveYOLOv8(cfg='yolov8n.yaml', ch=3, nc=nc)

        if Path(self.pretrained).exists():
            LOGGER.info(f'Loading pretrained weights from {self.pretrained}')
            pretrained = YOLO(self.pretrained)
            pretrained_dict = pretrained.model.state_dict()
            model_dict = self.model.state_dict()
            pretrained_dict = {k: v for k, v in pretrained_dict.items()
                             if k in model_dict and 'domain_classifier' not in k and 'cbam' not in k}
            model_dict.update(pretrained_dict)
            self.model.load_state_dict(model_dict, strict=False)
            LOGGER.info('Loaded pretrained weights (excluding CBAM and domain classifier)')

        self.model = self.model.to(self.device)
        self.model.train()

    def _build_loader(self, data_cfg, split='train', mode='train', rect=False):
        from ultralytics.cfg import get_cfg
        cfg = get_cfg()
        cfg.imgsz = self.img_size
        img_path = str(Path(data_cfg['path']) / data_cfg[split])
        dataset = build_yolo_dataset(cfg, img_path, self.batch_size, data_cfg, mode=mode, rect=rect)
        shuffle = mode == 'train'
        return build_dataloader(dataset, self.batch_size, self.workers, shuffle=shuffle)

    def setup_dataloaders(self):
        LOGGER.info('Setting up dataloaders...')
        self.source_loader = self._build_loader(self.source_cfg, split='train', mode='train')
        self.target_loader = self._build_loader(self.target_cfg, split='train', mode='train')
        self.val_loader = self._build_loader(self.source_cfg, split='val', mode='val', rect=True)
        LOGGER.info(f'Source batches: {len(self.source_loader)}, Target batches: {len(self.target_loader)}')

    def setup_training(self):
        self.optimizer = optim.Adam([
            {'params': [p for n, p in self.model.named_parameters()
                       if 'domain_classifier' not in n], 'lr': self.lr},
            {'params': self.model.domain_classifier.parameters(), 'lr': self.lr}
        ])

        if HAS_DETECTION_LOSS:
            self.detection_criterion = v8DetectionLoss(self.model)
            LOGGER.info('Using v8DetectionLoss')
        else:
            self.detection_criterion = None

        self.domain_criterion = nn.BCEWithLogitsLoss()
        self.best_score = 0.0
        self.history = {'det_loss': [], 'dom_loss': [], 'total_loss': [], 'alpha': [], 'val_score': []}

    def _batch_to_device(self, batch):
        """Move batch label tensors to training device."""
        for k in ('batch_idx', 'cls', 'bboxes'):
            if k in batch and isinstance(batch[k], torch.Tensor):
                batch[k] = batch[k].to(self.device, non_blocking=True)

    def compute_detection_loss(self, predictions, batch):
        if self.detection_criterion is not None:
            loss, loss_items = self.detection_criterion(predictions, batch)
            return loss.sum(), loss_items
        else:
            return torch.tensor(0.0, device=self.device), None

    def train_epoch(self, epoch):
        self.model.train()
        progress = epoch / self.epochs
        alpha = (2.0 / (1.0 + torch.exp(torch.tensor(-10 * progress, device=self.device))) - 1).item()

        pbar = tqdm(
            zip(self.source_loader, self.target_loader),
            total=min(len(self.source_loader), len(self.target_loader)),
            desc=f'Epoch {epoch+1}/{self.epochs}'
        )

        epoch_loss = epoch_det = epoch_dom = 0
        n_batches = 0

        for source_batch, target_batch in pbar:
            source_imgs = source_batch['img'].to(self.device, non_blocking=True).float() / 255.0
            target_imgs = target_batch['img'].to(self.device, non_blocking=True).float() / 255.0
            self._batch_to_device(source_batch)

            self.optimizer.zero_grad()

            source_preds, source_domain_pred = self.model(source_imgs, alpha=alpha, return_domain=True)
            _, target_domain_pred = self.model(target_imgs, alpha=alpha, return_domain=True)

            det_loss, _ = self.compute_detection_loss(source_preds, source_batch)

            dom_loss = (
                self.domain_criterion(source_domain_pred, torch.ones_like(source_domain_pred)) +
                self.domain_criterion(target_domain_pred, torch.zeros_like(target_domain_pred))
            ) / 2.0

            total_loss = det_loss + dom_loss
            total_loss.backward()
            self.optimizer.step()

            epoch_loss += total_loss.item()
            epoch_det += det_loss.item()
            epoch_dom += dom_loss.item()
            n_batches += 1

            pbar.set_postfix({
                'loss': f'{total_loss.item():.4f}',
                'det': f'{det_loss.item():.4f}',
                'dom': f'{dom_loss.item():.4f}',
                'alpha': f'{alpha:.3f}'
            })

        n = max(n_batches, 1)
        metrics = {
            'loss': epoch_loss / n,
            'det': epoch_det / n,
            'dom': epoch_dom / n,
            'alpha': alpha
        }
        self.history['total_loss'].append(metrics['loss'])
        self.history['det_loss'].append(metrics['det'])
        self.history['dom_loss'].append(metrics['dom'])
        self.history['alpha'].append(alpha)
        return metrics

    @torch.no_grad()
    def validate(self):
        self.model.eval()
        total_loss = 0.0
        n_batches = 0
        for batch in tqdm(self.val_loader, desc='Validating'):
            imgs = batch['img'].to(self.device, non_blocking=True).float() / 255.0
            self._batch_to_device(batch)
            preds = self.model(imgs)
            try:
                det_loss, _ = self.compute_detection_loss(preds, batch)
                total_loss += det_loss.item()
            except Exception as e:
                LOGGER.warning(f'Validation loss computation failed: {e}')
            n_batches += 1
        avg_loss = total_loss / max(n_batches, 1)
        score = 1.0 / (1.0 + avg_loss)
        self.history['val_score'].append(score)
        LOGGER.info(f'Validation: avg_loss={avg_loss:.4f}, score={score:.4f}')
        return score

    def save_checkpoint(self, epoch, is_best=False):
        checkpoint = {
            'epoch': epoch,
            'model': de_parallel(self.model).state_dict(),
            'optimizer': self.optimizer.state_dict(),
            'best_score': self.best_score,
        }
        torch.save(checkpoint, self.save_dir / 'last.pt')
        if is_best:
            torch.save(checkpoint, self.save_dir / 'best.pt')
            LOGGER.info(f'New best model saved!')
        if (epoch + 1) % 10 == 0:
            torch.save(checkpoint, self.save_dir / f'epoch_{epoch+1}.pt')

    def train(self):
        LOGGER.info(f'Training for {self.epochs} epochs on {self.device}')
        LOGGER.info(f'Batch size: {self.batch_size}, LR: {self.lr}')

        for epoch in range(self.epochs):
            metrics = self.train_epoch(epoch)
            LOGGER.info(
                f"Epoch {epoch+1}/{self.epochs} - "
                f"Loss: {metrics['loss']:.4f}, Det: {metrics['det']:.4f}, "
                f"Dom: {metrics['dom']:.4f}, Alpha: {metrics['alpha']:.3f}"
            )

            if (epoch + 1) % self.val_interval == 0:
                val_score = self.validate()
                is_best = val_score > self.best_score
                if is_best:
                    self.best_score = val_score
                self.save_checkpoint(epoch, is_best=is_best)
            else:
                self.save_checkpoint(epoch)

        LOGGER.info(f'Training complete! Best score: {self.best_score:.4f}')
        return self.history

## 4. Download Dataset from Roboflow & Prepare Domains

Downloads the NEU Surface Defect dataset directly, then splits it into source (original) and target (augmented) domains.

In [ ]:
from roboflow import Roboflow

# NEU-DET: Balanced steel defect dataset (300 images/class x 6 classes = 1800 total)
# Classes: crazing, inclusion, patches, pitted_surface, rolled-in_scale, scratches

# Use Colab Secrets (recommended) or fall back to manual entry
# To set up: click the key icon in the left sidebar → Add secret "ROBOFLOW_API_KEY"
import os
try:
    from google.colab import userdata
    api_key = userdata.get('ROBOFLOW_API_KEY')
    print("Using API key from Colab Secrets")
except (ImportError, userdata.SecretNotFoundError):
    api_key = os.environ.get('ROBOFLOW_API_KEY', '')
    if not api_key:
        api_key = input("Enter your Roboflow API key: ")

rf = Roboflow(api_key=api_key)
project = rf.workspace("neudatasetoriginal").project("neu-steel-defect-dataset")
version = project.version(1)
dataset = version.download("yolov8", location="/content/neu_dataset")

DATASET_ROOT = "/content/neu_dataset"
print(f"Dataset downloaded to: {DATASET_ROOT}")

# Verify balance
from pathlib import Path
label_dir = Path(DATASET_ROOT) / "train" / "labels"
if label_dir.exists():
    from collections import Counter
    class_counts = Counter()
    for lbl_file in label_dir.glob("*.txt"):
        for line in open(lbl_file):
            cls_id = int(line.strip().split()[0])
            class_counts[cls_id] += 1
    print(f"\nClass distribution (train):")
    for cls_id in sorted(class_counts):
        print(f"  Class {cls_id}: {class_counts[cls_id]} annotations")
    print(f"  Total: {sum(class_counts.values())} annotations")

In [ ]:
import os, shutil, random, cv2, yaml
import numpy as np
from pathlib import Path
from tqdm import tqdm

def augment_target_domain(img):
    """Simulate a different camera/lighting environment."""
    img = img.astype(np.float32)
    img *= random.uniform(0.5, 0.7)  # darken
    img[:, :, 0] = np.clip(img[:, :, 0] * random.uniform(1.1, 1.3), 0, 255)  # blue tint
    img[:, :, 2] = np.clip(img[:, :, 2] * random.uniform(0.8, 0.9), 0, 255)  # reduce red
    noise = np.random.normal(0, random.uniform(8, 15), img.shape)
    img = np.clip(img + noise, 0, 255)
    if random.random() > 0.5:
        img = cv2.GaussianBlur(img.astype(np.uint8), (3, 3), 0).astype(np.float32)
    return img.astype(np.uint8)

def setup_experiment(dataset_root, output_root='/content/datasets', target_ratio=0.4):
    dataset_root = Path(dataset_root)
    output_root = Path(output_root)
    source_dir = output_root / 'source'
    target_dir = output_root / 'target'

    # Auto-detect Roboflow folder structure
    # Roboflow downloads to: dataset_root/train/images, dataset_root/valid/images, etc.
    # Some versions use 'test' instead of 'valid'
    train_img_dir = dataset_root / 'train' / 'images'
    if not train_img_dir.exists():
        # Try flat structure
        train_img_dir = dataset_root / 'train'
    
    valid_img_dir = dataset_root / 'valid' / 'images'
    if not valid_img_dir.exists():
        valid_img_dir = dataset_root / 'test' / 'images'
    if not valid_img_dir.exists():
        valid_img_dir = dataset_root / 'valid'

    print(f"Train images dir: {train_img_dir}")
    print(f"Valid images dir: {valid_img_dir}")

    train_images = sorted(list(train_img_dir.glob('*.jpg')) + list(train_img_dir.glob('*.png')))
    random.seed(42)
    random.shuffle(train_images)
    split_idx = int(len(train_images) * (1 - target_ratio))
    source_imgs = train_images[:split_idx]
    target_imgs = train_images[split_idx:]

    print(f'Total: {len(train_images)} | Source: {len(source_imgs)} | Target: {len(target_imgs)}')

    # Create dirs
    for d in [source_dir/'train'/'images', source_dir/'train'/'labels',
              source_dir/'val'/'images', source_dir/'val'/'labels',
              target_dir/'train'/'images', target_dir/'train'/'labels']:
        d.mkdir(parents=True, exist_ok=True)

    # Helper to find label for an image
    def find_label(img_path):
        lbl = img_path.parent.parent / 'labels' / (img_path.stem + '.txt')
        if lbl.exists():
            return lbl
        lbl = img_path.with_suffix('.txt')
        if lbl.exists():
            return lbl
        return None

    # Source train
    for img_path in tqdm(source_imgs, desc='Source train'):
        shutil.copy2(img_path, source_dir / 'train' / 'images' / img_path.name)
        lbl = find_label(img_path)
        if lbl:
            shutil.copy2(lbl, source_dir / 'train' / 'labels' / (img_path.stem + '.txt'))

    # Source val
    val_images = sorted(list(valid_img_dir.glob('*.jpg')) + list(valid_img_dir.glob('*.png')))
    for img_path in tqdm(val_images, desc='Source val'):
        shutil.copy2(img_path, source_dir / 'val' / 'images' / img_path.name)
        lbl = find_label(img_path)
        if lbl:
            shutil.copy2(lbl, source_dir / 'val' / 'labels' / (img_path.stem + '.txt'))

    # Target train (augmented)
    for img_path in tqdm(target_imgs, desc='Target (augmenting)'):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        cv2.imwrite(str(target_dir / 'train' / 'images' / img_path.name), augment_target_domain(img))
        lbl = find_label(img_path)
        if lbl:
            shutil.copy2(lbl, target_dir / 'train' / 'labels' / (img_path.stem + '.txt'))

    # Read class names from Roboflow's data.yaml if available
    roboflow_yaml = dataset_root / 'data.yaml'
    if roboflow_yaml.exists():
        with open(roboflow_yaml) as f:
            rf_cfg = yaml.safe_load(f)
        class_names = rf_cfg.get('names', [])
        nc = rf_cfg.get('nc', len(class_names))
    else:
        class_names = ['crazing','crease','crescent_gap','inclusion','oil_spot',
                       'patches','pitted_surface','punching_hole','rolled-in_scale',
                       'rolled_pit','scratches','silk_spot','waist_folding','water_spot','welding_line']
        nc = len(class_names)

    print(f"Classes ({nc}): {class_names}")

    # Write YAMLs
    for name, d, val in [('source', source_dir, 'val/images'), ('target', target_dir, 'train/images')]:
        with open(output_root / f'{name}.yaml', 'w') as f:
            yaml.dump({'path': str(d.resolve()), 'train': 'train/images', 'val': val,
                       'nc': nc, 'names': class_names}, f)

    print(f'\nDone! Configs: {output_root}/source.yaml, {output_root}/target.yaml')
    return str(output_root / 'source.yaml'), str(output_root / 'target.yaml'), nc

source_yaml, target_yaml, NUM_CLASSES = setup_experiment(DATASET_ROOT)

## 5. Smoke Test

In [ ]:
import torch
from model import DomainAdaptiveYOLOv8, CBAMC2f

model = DomainAdaptiveYOLOv8(cfg='yolov8n.yaml', ch=3, nc=NUM_CLASSES)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
x = torch.randn(2, 3, 640, 640, device=device)

# Inference
model.eval()
with torch.no_grad():
    out = model(x)
print(f'Inference OK — output type: {type(out).__name__}')

# Training + domain
model.train()
out, dom = model(x, alpha=0.5, return_domain=True)
print(f'Domain pred shape: {dom.shape}')
dom.sum().backward()
print(f'Backward OK')
print(f'CBAM modules: {sum(1 for m in model.modules() if isinstance(m, CBAMC2f))}')
print(f'Domain classifier: {model.domain_classifier is not None}')
del model; torch.cuda.empty_cache()

## 6. Train!

In [ ]:
from train import DomainAdaptiveTrainer

trainer = DomainAdaptiveTrainer(
    source_data=source_yaml,
    target_data=target_yaml,
    epochs=50,           # increase for better results
    batch_size=16,       # T4 can handle 16 at 640px
    img_size=640,
    device='0',          # GPU
    workers=2,
    save_dir='runs/dann_cbam',
    pretrained='yolov8n.pt',
    lr=0.001,
    val_interval=5,
)

history = trainer.train()

## 7. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curves
axes[0].plot(history['total_loss'], label='Total Loss', linewidth=2)
axes[0].plot(history['det_loss'], label='Detection Loss', linewidth=2)
axes[0].plot(history['dom_loss'], label='Domain Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Losses')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Alpha schedule
axes[1].plot(history['alpha'], color='red', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Alpha')
axes[1].set_title('DANN Alpha Schedule (Gradient Reversal Strength)')
axes[1].grid(True, alpha=0.3)

# Validation score
if history['val_score']:
    val_epochs = list(range(trainer.val_interval-1, trainer.epochs, trainer.val_interval))
    axes[2].plot(val_epochs[:len(history['val_score'])], history['val_score'], 'go-', linewidth=2, markersize=8)
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Score')
    axes[2].set_title('Validation Score')
    axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('runs/dann_cbam/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to runs/dann_cbam/training_curves.png')

## 8. Visualize Domain Adaptation Effect

Compare source vs target domain sample images and see what DANN is adapting.

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import random

source_imgs = sorted(Path('/content/datasets/source/train/images').glob('*.jpg'))
target_imgs = sorted(Path('/content/datasets/target/train/images').glob('*.jpg'))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Source Domain (top) vs Target Domain (bottom)', fontsize=16)

for i in range(4):
    idx = random.randint(0, min(len(source_imgs), len(target_imgs)) - 1)
    src = cv2.cvtColor(cv2.imread(str(source_imgs[idx])), cv2.COLOR_BGR2RGB)
    tgt = cv2.cvtColor(cv2.imread(str(target_imgs[idx])), cv2.COLOR_BGR2RGB)
    axes[0][i].imshow(src)
    axes[0][i].set_title(f'Source #{idx}')
    axes[0][i].axis('off')
    axes[1][i].imshow(tgt)
    axes[1][i].set_title(f'Target #{idx}')
    axes[1][i].axis('off')

plt.tight_layout()
plt.show()

## 9. Run Inference with Trained Model

In [ ]:
import torch
from model import DomainAdaptiveYOLOv8

# Load best checkpoint (weights_only=True prevents arbitrary code execution)
model = DomainAdaptiveYOLOv8(cfg='yolov8n.yaml', ch=3, nc=NUM_CLASSES)
ckpt = torch.load('runs/dann_cbam/best.pt', map_location='cuda', weights_only=True)
model.load_state_dict(ckpt['model'])
model = model.to('cuda').eval()

print(f'Loaded best model from epoch {ckpt["epoch"]+1}, score: {ckpt["best_score"]:.4f}')

# Test on a target domain image
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

test_img_path = sorted(Path('/content/datasets/target/train/images').glob('*.jpg'))[0]
img = cv2.imread(str(test_img_path))
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
input_tensor = torch.from_numpy(img_rgb).permute(2, 0, 1).unsqueeze(0).float() / 255.0
input_tensor = torch.nn.functional.interpolate(input_tensor, size=(640, 640))
input_tensor = input_tensor.to('cuda')

with torch.no_grad():
    preds = model(input_tensor)

print(f'Prediction type: {type(preds).__name__}')
plt.figure(figsize=(8, 8))
plt.imshow(img_rgb)
plt.title('Target domain inference')
plt.axis('off')
plt.show()

## 10. Download Results

In [ ]:
# Zip results for download
!zip -r /content/dann_cbam_results.zip runs/dann_cbam/

from google.colab import files
files.download('/content/dann_cbam_results.zip')